# AutoGSQ-RCO — Qwen3-4B full pipeline on Colab

End-to-end: `generate-db` (36 layers) → `allocate` → `assemble` → PPL eval.
Forked from the local Windows pipeline so the RTX A4000 stays free.

**Storage strategy (read this):** the candidate DB is ~60 GB (measured
~1.7 GB/layer locally) — too big for a small Google Drive, so it lives on
Colab's **local disk** (`/content/autogsq_rco`, ~110 GB). Local disk is
**wiped when the session dies**, so finished layers are synced periodically
to a Hugging Face dataset repo (cell 8) and restored with one cell (7b)
after a reconnect. Drive is used only for the final ~1.4 GB GGUF, if at all.

**Session math:** free-tier runs cap at ~4-5h. If build pace (watch files
appearing in the DB dir) projects past the cap, sync to Hub, let the
session die, then restore + resume in a fresh one — resume skips finished
layers, so multi-session completion works.


In [ ]:
# 0 — GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
# 1 — Paths live on LOCAL disk (roomy, but ephemeral — see header).
# Drive mount is optional: only needed if you want the final GGUF copied
# there. Skip it otherwise.
ROOT = '/content/autogsq_rco'
DB = f'{ROOT}/qwen3_4b_db'
!mkdir -p "$ROOT"
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted (optional backup target)')
except Exception as e:
    print(f'no Drive mount ({e}); continuing local-only')


In [ ]:
# 2 — Clone + install. Push this repo to GitHub first, then store its URL
# in Colab Secrets as REPO_URL (never paste URLs/tokens into the notebook).
# NOTE: `datasets` is required for wikitext2 calibration but is not
# declared in pyproject.toml, so it is pinned explicitly here.
try:
    from google.colab import userdata
    _url = userdata.get('REPO_URL')
except Exception:
    _url = None
if not _url:
    raise SystemExit('Set REPO_URL in Colab Secrets, then re-run this cell.')
import os
%cd /content
if os.path.isdir("AutoGSQ-RCO"):
    %cd AutoGSQ-RCO
    !git pull --ff-only
else:
    !git clone {_url} AutoGSQ-RCO
    %cd AutoGSQ-RCO
!pip install -q -e . "datasets>=3.0.0"


In [ ]:
# 3 — Optional HF login (raises quotas; harmless if skipped). Token stored via
# Colab Secrets as HF_TOKEN, never pasted into the notebook.
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'))
    print('HF login OK')
except Exception as e:
    print(f'skipping HF login ({e}); continuing unauthenticated')

In [ ]:
# 4 — Colab config. Same hyperparams as the local probe (runtime/qwen3_4b_probe.yaml),
# except model.name is the HF hub ID (loader takes hub IDs directly) and all
# outputs point at Drive. `device: cuda` = the Colab GPU.
config = '''model:
  name: "Qwen/Qwen3-4B"
  device: "cuda"
  dtype: "bfloat16"

data:
  dataset_name: "wikitext2"
  num_samples: 64
  max_length: 1024

gsq:
  bitwidth_options: [2, 3, 4, "ternary"]
  init_method: "gptq"
  group_size: 128
  temperature: [2.0, 0.05]
  logit_scale: [100.0, 500.0]
  num_epochs: 4
  optimizer: "lion"
  learning_rate: 0.001
  steps_per_epoch: 32
  lr_logits: 0.0002
  lr_scales: 0.0001
  gs_weight_decay: 1.0
  warmup_ratio: 0.1
  logits_dtype: "bfloat16"
  gptq_nsamples: 64
  gptq_damping: 0.01
  calib_max_length: 1024
  calib_device: "cuda"

rco:
  target_bpw: 2.75
  eval_dataset: "wikitext2"
  num_steps: 300
  lr: 0.01
  temperature: 1.0
  temperature_min: 0.1
  temperature_schedule: "linear"
  refine_steps: 20
  refine_batches: 2
  refine_max_length: 256

gguf:
  gguf_type_tolerance_bpw: 0.15
  fallback_on_no_match: "round_down"

output:
  checkpoint_dir: "/content/autogsq_rco/qwen3_4b_colab"
  run_id: null
'''
open('/content/qwen3_4b_colab.yaml', 'w').write(config)
print('wrote /content/qwen3_4b_colab.yaml')


In [ ]:
# 5b — Preflight: the full DB needs ~60 GB. Abort early if it won't fit.
import shutil
free = shutil.disk_usage('/content').free / 1e9
print(f'free on /content: {free:.1f} GB (need ~65 GB headroom)')
assert free > 65, 'Not enough local disk — Runtime → Factory reset runtime, then re-run from cell 0.'


In [ ]:
# 5c — Restore previous layers from Hub (skip on first run).
# One-time setup: create an EMPTY dataset repo on HF and put its name here.
DB_REPO = "Egalitaristen/autogsq-qwen3-4b-db"  # <-- edit me
import os
if os.path.isfile(DB + '/progress.json'):
    print('DB already present locally; skipping restore')
else:
    try:
        from huggingface_hub import snapshot_download
        snapshot_download(repo_id=DB_REPO, repo_type='dataset', local_dir=DB)
        print('restored from Hub; resume will skip finished layers')
    except Exception as e:
        print('nothing to restore; starting fresh (%s)' % e)


In [ ]:
# 5 — Stage 1: candidate DB build. THE long cell (hours). Resume is on by
# default: if Colab kills the session, reconnect + re-run from cell 1,
# then re-run THIS cell; finished layers print
# "already complete, skipping" and it continues where it stopped.
# Smoke-test first with `--max-layers 2` (~20 min) before the full run.
!autogsq generate-db --config /content/qwen3_4b_colab.yaml --out-dir "$ROOT/qwen3_4b_db"

In [ ]:
# 5d — Sync finished layers to Hub. RE-RUN THIS CELL HOURLY while the
# build runs (it only uploads new/changed files). Needs HF_TOKEN with
# WRITE access. If the session dies, restore with cell 5c in a fresh one.
DB_REPO = "Egalitaristen/autogsq-qwen3-4b-db"  # same name as cell 5c
!huggingface-cli upload {DB_REPO} "$ROOT/qwen3_4b_db" --repo-type dataset 2>&1 | tail -2


In [ ]:
# 6 — Progress check (cheap, re-run anytime)
!autogsq status --db-dir "$ROOT/qwen3_4b_db"

In [ ]:
# 7 — Stage 2: RCO allocation at 2.75 bpw (minutes, CPU)
!autogsq allocate --db-dir "$ROOT/qwen3_4b_db" --config /content/qwen3_4b_colab.yaml --target-bpw 2.75 --out "$ROOT/qwen3_4b_alloc_full.json"

In [ ]:
# 8 — Stage 3: assemble the GGUF. Colab VMs have 2 vCPUs, so -j 2.
# (~15-20 min on 16 local cores; expect longer here.)
!autogsq assemble --db-dir "$ROOT/qwen3_4b_db" --allocation "$ROOT/qwen3_4b_alloc_full.json" --output "$ROOT/qwen3_4b_colab/model-full.gguf" --config /content/qwen3_4b_colab.yaml -j 2

In [ ]:
# 9 — Stage 4: PPL eval. --source takes the hub ID directly (lazy weight
# loading, no local copy needed). Local baseline for comparison: 14.21
# (official Q4_K_M). Our local RCO number at probe scale: 42.67.
!python src/autogsq/eval/gguf_ppl.py --gguf "$ROOT/qwen3_4b_colab/model-full.gguf" --source Qwen/Qwen3-4B --max-windows 20

In [ ]:
# 10 — Artifacts
!ls -lh "$ROOT/qwen3_4b_colab/" "$ROOT/qwen3_4b_alloc_full.json"